# Task 7 Responsible and Sustainable AI

I would assess the traffic models for uneven prediction errors across time and weather groups, data coverage limitations and responsible use.

The dataset does not contain demographic attributes, so these checks cannot establish demographic fairness.

I would also document human oversight, model governance and ways to reduce unnecessary training and storage.

### Configure responsible AI logging

I would configure the execution log and load the saved test predictions for the group error analysis.

In [1]:
import logging
from pathlib import Path


import numpy as np
import pandas as pd

PART3_DIR = Path.cwd().parent
REPO_DIR = PART3_DIR.parent



logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            PART3_DIR / "part3_smart_city_traffic_responsible_ai.log",
            encoding="utf-8"
        )
    ],
    force=True
)


logger = logging.getLogger(__name__)

logger.info("Responsible and sustainable AI notebook has started.")


test_predictions = pd.read_csv(
    PART3_DIR / "results" / "task3_test_predictions.csv",
    parse_dates=["date_time"]
)



required_columns = [
    "date_time",
    "actual_traffic_volume",
    "neural_network_prediction",
    "random_forest_prediction"
]


if not set(required_columns).issubset(test_predictions.columns):
    raise ValueError("Required test prediction columns are missing.")
    

if test_predictions.empty:
    raise ValueError("Test predictions are empty.")

assert test_predictions["date_time"].notna().all()
assert np.isfinite(
    test_predictions[required_columns[1:]].to_numpy()
).all()

display(pd.DataFrame([{
    "test_records": len(test_predictions),
    "first_date": test_predictions["date_time"].min(),
    "last_date": test_predictions["date_time"].max()
}]))


logger.info("Loaded and checked the saved held-out test predictions.")

2026-10-11 22:21:04,962 | INFO | __main__ | Responsible and sustainable AI notebook has started.


,test_records,first_date,last_date
0,4944,2018-04-14 16:00:00,2018-09-30 23:00:00


2026-10-11 22:21:05,044 | INFO | __main__ | Loaded and checked the saved held-out test predictions.


### Compare prediction errors across time groups

I would compare both regression models across night, morning, afternoon and evening records in the held-out test set.

For each group, I would report record count, mean absolute error and mean signed error. 

These checks describe uneven performance across time groups; they do not establish demographic fairness.

In [2]:
error_analysis = test_predictions.copy()

error_analysis["time_group"] = pd.cut(
    error_analysis["date_time"].dt.hour,
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"]
)


group_error_records = []


for model_name, prediction_column in [
    ("Random Forest", "random_forest_prediction"),
    ("Neural Network", "neural_network_prediction")
]:
    for group_name, group in error_analysis.groupby(
        "time_group",
        observed=True
    ):
        errors = (
            group[prediction_column]
            - group["actual_traffic_volume"]
        )

        group_error_records.append({
            "model": model_name,
            "time_group": str(group_name),
            "records": len(group),
            "mean_actual_volume": group[
                "actual_traffic_volume"
            ].mean(),
            "MAE": errors.abs().mean(),
            "mean_signed_error": errors.mean()
        })

time_group_errors = pd.DataFrame(group_error_records)


display(
    time_group_errors.style.hide(axis="index").format({
        "mean_actual_volume": "{:.2f}",
        "MAE": "{:.2f}",
        "mean_signed_error": "{:.2f}"
    })
)

results_dir = PART3_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

time_group_errors.to_csv(
    results_dir / "responsible_ai_time_group_errors.csv",
    index=False
)


logger.info(
    "Compared and saved held-out prediction errors across time groups."
)

model,time_group,records,mean_actual_volume,MAE,mean_signed_error
Random Forest,Night,1267,874.31,97.14,11.10
Random Forest,Morning,1298,4497.40,330.02,-22.97
Random Forest,Afternoon,1187,5101.27,318.96,16.09
Random Forest,Evening,1192,2848.49,260.14,43.28
Neural Network,Night,1267,874.31,116.40,30.09
Neural Network,Morning,1298,4497.40,324.96,76.02
Neural Network,Afternoon,1187,5101.27,319.63,18.74
Neural Network,Evening,1192,2848.49,269.82,47.06


2026-10-11 22:28:20,921 | INFO | __main__ | Compared and saved held-out prediction errors across time groups.


### Interpret time group performance

Random Forest MAE ranged from 97.14 vehicles at night to 330.02 in the morning. 

Random Forest slightly underpredicted morning volume on average, with a signed error of -22.97 vehicles. The neural network overpredicted morning volume by 76.02 vehicles on average.

Each time group contained between 1187 and 1298 test records. 

These descriptive results identify groups for further investigation. They do not establish demographic fairness or explain the causes of the error differences. 

### Compare weekday and weekend performance

I would compare prediction errors and record counts for weekdays and weekends.

In [3]:
error_analysis["day_group"] = np.where(
    error_analysis["date_time"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)


day_error_records = []


for model_name, prediction_column in [
    ("Random Forest", "random_forest_prediction"),
    ("Neural Network", "neural_network_prediction")
]:
    for group_name, group in error_analysis.groupby("day_group"):
        errors = (
            group[prediction_column]
            - group["actual_traffic_volume"]
        )

        day_error_records.append({
            "model": model_name,
            "day_group": group_name,
            "records": len(group),
            "mean_actual_volume": group[
                "actual_traffic_volume"
            ].mean(),
            "MAE": errors.abs().mean(),
            "mean_signed_error": errors.mean()
        })
        

day_group_errors = pd.DataFrame(day_error_records)

display(
    day_group_errors.style.hide(axis="index").format({
        "mean_actual_volume": "{:.2f}",
        "MAE": "{:.2f}",
        "mean_signed_error": "{:.2f}"
    })
)


day_group_errors.to_csv(
    results_dir / "responsible_ai_day_group_errors.csv",
    index=False
)


logger.info(
    "Compared and saved held-out prediction errors for weekdays and weekends."
)

model,day_group,records,mean_actual_volume,MAE,mean_signed_error
Random Forest,Weekday,3517,3604.24,222.63,-0.99
Random Forest,Weekend,1427,2606.78,320.35,40.94
Neural Network,Weekday,3517,3604.24,234.90,76.20
Neural Network,Weekend,1427,2606.78,311.25,-37.03


2026-10-11 22:32:21,690 | INFO | __main__ | Compared and saved held-out prediction errors for weekdays and weekends.


### Interpret weekday and weekend performance

Random Forest MAE was 222.63 vehicles on weekdays and 320.35 on weekends. Neural Network MAE was 234.90 on weekdays and 311.25 on weekends.

Average actual volume was 3604.24 vehicles on weekdays and 2606.78 on weekends. 

Random Forest average signed error was -0.99 on weekdays and +40.94 on weekends. Neural Network average signed error was +76.20 on weekdays and -37.03 on weekends. 

The test set contained 3517 weekday records and 1427 weekend records. Weekend performance warrants further investigation, but these descriptive comparisons do not establish the cause of the difference or demographic unfairness.

### Align weather labels with held out predictions

I would repeat the original cleaning and chronological split, then verify that test timestamps and actual traffic volumes match the saved prediction rows before attaching weather labels.

Multiple records can share a timestamp, so joining on timestamp alone could duplicate rows.

In [4]:
import sys

part2_path = REPO_DIR / "capstone_part2"
if str(part2_path) not in sys.path:
    sys.path.insert(0, str(part2_path))

from pipeline import (
    validate_schema,
    standardise_categories,
    clean_datetime,
    convert_numeric_columns,
    fill_missing_categories,
    clean_other_numbers,
    remove_duplicates
)


raw_matches = list(
    (REPO_DIR / "part1_data_analytics").rglob(
        "Metro_Interstate_Traffic_Volume.csv"
    )
)


if len(raw_matches) != 1:
    raise FileNotFoundError(
        "Could not uniquely locate the raw traffic dataset."
    )
    

coverage_data = pd.read_csv(
    raw_matches[0],
    keep_default_na=False
)
validate_schema(coverage_data)

for cleaning_function in [
    standardise_categories,
    clean_datetime,
    convert_numeric_columns,
    fill_missing_categories,
    clean_other_numbers,
    remove_duplicates
]:
    coverage_data = cleaning_function(coverage_data)


coverage_data = (
    coverage_data.sort_values("date_time", kind="stable")
    .reset_index(drop=True)
)

unique_times = (
    coverage_data["date_time"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

test_start = unique_times.iloc[int(len(unique_times) * 0.90)]

aligned_test_data = coverage_data.loc[
    coverage_data["date_time"] >= test_start
].reset_index(drop=True)

assert len(aligned_test_data) == len(test_predictions)

pd.testing.assert_series_equal(
    aligned_test_data["date_time"],
    test_predictions["date_time"].reset_index(drop=True),
    check_names=False
)

np.testing.assert_allclose(
    aligned_test_data["traffic_volume"],
    test_predictions["actual_traffic_volume"],
    rtol=0,
    atol=0
)

error_analysis["weather_main"] = (
    aligned_test_data["weather_main"].to_numpy()
)


logger.info(
    "Verified test row alignment and attached weather labels."
)

2026-10-11 22:36:46,860 | INFO | pipeline | Column check has passed. All required columns are present.
2026-10-11 22:36:46,945 | INFO | pipeline | holiday: no text changes are required.
2026-10-11 22:36:46,978 | INFO | pipeline | weather_main: no text changes are required.
2026-10-11 22:36:47,009 | WARNING | pipeline | weather_description: changed 1730 rows to fix spaces and capital letters.
2026-10-11 22:36:47,070 | INFO | pipeline | Date check has passed. No missing or invalid dates.
2026-10-11 22:36:47,071 | INFO | pipeline | date_time is now in datetime format.
2026-10-11 22:36:47,083 | INFO | pipeline | temp: number conversion finished.
2026-10-11 22:36:47,087 | INFO | pipeline | rain_1h: number conversion finished.
2026-10-11 22:36:47,091 | INFO | pipeline | snow_1h: number conversion finished.
2026-10-11 22:36:47,095 | INFO | pipeline | clouds_all: number conversion finished.
2026-10-11 22:36:47,099 | INFO | pipeline | traffic_volume: number conversion finished.
2026-10-11 22:36

### Compare weather group errors and coverage

I would compare both models across weather categories and report the number of test records in each category.

Groups with fewer than 100 records would be marked as limited evidence. This is a reporting rule for this analysis, not a statistical guarantee. 

In [5]:
weather_error_records = []


for model_name, prediction_column in [
    ("Random Forest", "random_forest_prediction"),
    ("Neural Network", "neural_network_prediction")
]:
    for weather_name, group in error_analysis.groupby("weather_main"):
        errors = (
            group[prediction_column]
            - group["actual_traffic_volume"]
        )

        
        weather_error_records.append({
            "model": model_name,
            "weather": weather_name,
            "records": len(group),
            "mean_actual_volume": group[
                "actual_traffic_volume"
            ].mean(),
            "MAE": errors.abs().mean(),
            "mean_signed_error": errors.mean(),
            "evidence": (
                "Limited sample" if len(group) < 100
                else "At least 100 records"
            )
        })

weather_group_errors = pd.DataFrame(weather_error_records)

display(
    weather_group_errors.style.hide(axis="index").format({
        "mean_actual_volume": "{:.2f}",
        "MAE": "{:.2f}",
        "mean_signed_error": "{:.2f}"
    })
)

weather_group_errors.to_csv(
    results_dir / "responsible_ai_weather_group_errors.csv",
    index=False
)

weather_coverage = pd.DataFrame({
    "weather": sorted(coverage_data["weather_main"].unique())
})

weather_coverage["test_records"] = (
    weather_coverage["weather"]
    .map(aligned_test_data["weather_main"].value_counts())
    .fillna(0)
    .astype(int)
)

weather_coverage["evidence"] = np.select(
    [
        weather_coverage["test_records"].eq(0),
        weather_coverage["test_records"].lt(100)
    ],
    ["No test records", "Limited sample"],
    default="At least 100 records"
)

display(weather_coverage.style.hide(axis="index"))

weather_coverage.to_csv(
    results_dir / "responsible_ai_weather_coverage.csv",
    index=False
)


logger.info(
    "Saved weather group errors and identified limited or absent test coverage."
)

model,weather,records,mean_actual_volume,MAE,mean_signed_error,evidence
Random Forest,Clear,1363,2898.16,212.98,-3.47,At least 100 records
Random Forest,Clouds,1272,3992.89,247.93,-58.82,At least 100 records
Random Forest,Drizzle,237,3720.13,181.12,12.02,At least 100 records
Random Forest,Fog,112,2096.17,444.55,216.84,At least 100 records
Random Forest,Haze,115,3341.66,348.29,-38.04,At least 100 records
Random Forest,Mist,622,2853.80,245.33,41.64,At least 100 records
Random Forest,Rain,916,3444.70,242.50,12.13,At least 100 records
Random Forest,Smoke,2,4443.00,176.12,-176.12,Limited sample
Random Forest,Snow,47,1839.79,1125.13,1052.88,Limited sample
Random Forest,Thunderstorm,258,3257.13,285.84,99.19,At least 100 records


weather,test_records,evidence
Clear,1363,At least 100 records
Clouds,1272,At least 100 records
Drizzle,237,At least 100 records
Fog,112,At least 100 records
Haze,115,At least 100 records
Mist,622,At least 100 records
Rain,916,At least 100 records
Smoke,2,Limited sample
Snow,47,Limited sample
Squall,0,No test records


2026-10-11 22:38:43,760 | INFO | __main__ | Saved weather group errors and identified limited or absent test coverage.


### Interpret weather performance and coverage

Snow had the highest observed MAE: 1125.13 vehicles for Random Forest and 1164.69 for the neural network. Both models overpredicted snow traffic by approximately 1053 vehicles on average. 

Fog had 112 records, with MAE of 444.55 for Random Forest and 435.23 for the neural network. Both models also overpredicted fog traffic on average.

Smoke had only two test records, which is insufficient for a dependable comparison. Squall had no test records, so its held-out performance could not be assessed.

The test period ran from April to September 2018 and does not cover a complete annual weather cycle. 

These findings support collecting more representative severe-weather data and investigating errors before operational use. They do not establish demographic fairness or prove that weather alone caused the differences.

### Classification proxy and responsible use

The HighRisk classification target was constructed from traffic volume thresholds and selected weather conditions. 

Weather features also helped define the proxy label. Strong classification scores therefore demonstrate how well the models reproduce that rule; they do not establish accident prediction ability.

The proxy should not be used to declare a journey safe, issue penalties or make emergency decisions. Validating an accident-risk model would require suitable observed outcomes and a separate evaluation.

### Human oversight and model governance

Traffic recommendations should support human judgement. Operators should consider current incidents, road conditions and official information before acting on a recommendation.

Model versions, preparation settings, evaluation results and artifact hashes should be retained. Any replacement model should undergo validation and group performance review before deployment.

An ALERT should prompt investigation of data quality, seasonal changes and model performance. It should not automatically trigger retraining. A PASS result does not guarantee accurate predictions.

The project does not contain demographic attributes, so demographic fairness remains unassessed. 

### Sustainability and resource use

I havereused saved models for recommendation and deployment instead of retraining them for each task. Neural network early stopping completed training after 37 epochs, below the maximum of 100.

These choices avoided additional computation in this workflow. Energy consumption and carbon emissions were not measured, so I cannot quantify environmental savings or claim that one model is greener.

MLflow retained copies of model artifacts for reproducibility. This increases storage use. An operational project should define retention rules, review duplicate artifacts and archive obsolete versions while preserving required evidence.

Future model comparisons should measure training time, prediction latency, memory use and energy consumption where suitable tools are available. 

### Task 7 conclusions

Both regression models showed uneven errors across time, day and weather groups. Weekend errors were higher than weekday errors, and snow errors were large but supported by only 47 test records. 

The analysis identifies areas for investigation rather than establishing demographic fairness or explaining the causes of error differences. 

Responsible use requires representative data, human oversight, version control and continued performance review. Sustainability claims remain qualitative because energy and emissions were not measured.

### Save the responsible AI findings

I would save the group analysis findings, coverage limitations, proxy disclosure, governance recommendations and sustainability assessment as a separate report.

In [6]:
reports_dir = PART3_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)


responsible_ai_report = """# Responsible and Sustainable AI Report

## Scope

This analyss uses 4944 held-out test records from 14 April to
30 September 2018. It compares saved Random Forest and neural network
regression predictions without retraining or tuning on the test set.

The dataset contains no demographic attributes. 

## Time group performance

Random Forest MAE ranged from 97.14 vehicles at night to 330.02 in
the morning. Neural Network MAE ranged from 116.40 at night to 324.96
in the morning.

Traffic volume differs between time groups, so absolute error
differences require context and do not alone establish unfairness.

## Weekday and weekend performance

Random Forest MAE was 222.63 on weekdays and 320.35 on weekends.
Neural Network MAE was 234.90 on weekdays and 311.25 on weekends.

There were 3517 weekday and 1427 weekend records. Average actual volume
was 3604.24 on weekdays and 2606.78 on weekends. Higher weekend error
despite lower average volume warrants investigation.

## Weather performance and coverage

Snow MAE was 1125.13 for Random Forest and 1164.69 for the neural
network. Both models overpredicted snow traffic by approximately 1053
vehicles on average. Only 47 test records supported these estimates.

Fog had 112 records, with MAE 444.55 for Random Foest and 435.23 for
the neural network. Smoke had two records and squall had none.

The test period does not cover a complete annual weather cycle.
Multiple records can share an hourly timestamp, so record counts do
not represent independent observations. The fewer-than-100 reporting
rule flags limited evidence; it is not a statistical guarantee.

## Classification proxy

HighRisk was constructed from traffic volume thresholds and selected
weather conditions. It is not an observed accident or injury outcome.
Weather inputs also help define the label, so strong classification
scores demonstrate reproduction of the proxy rule rather than
validated accident prediction ability.

The proxy must not be used to declare journeys safe, issue penalties
or make emergency decisions.

## Human oversight and governance

Recommendations should support human judgement alongside current
incidents, road conditions and oficial information.

Retain model versions, preparation settings, evaluation results and
artifact hashes. Validate replacement models and review group errors
before deployment.

Investigate monitoring ALERT results for data quality, seasonal
changes and performance changes. Do not automatically retrain.
PASS does not guarantee accurate predictions.

Collect more representative severe-weather data and assess performance
across additional periods and corridors before wider use.

## Sustainability

Saved models were reused rather than retrained for recommendation and
deployment. Early stopping completed neural network training after
37 epochs instead of the maximum 100.

Energy and emissions were not measured. Environmental savings cannot
be quantified, and no claim is made that one model is greener.

MLflow artifact copies support reproducibility but increase storage.
Define retention rules and archive obsolete versions while preserving
required evidence.

Fuure comparisons should measure training time, inference latency,
memory use and, where feasible, energy consumption.

## Supporting results

- responsible_ai_time_group_errors.csv
- responsible_ai_day_group_errors.csv
- responsible_ai_weather_group_errors.csv
- responsible_ai_weather_coverage.csv

These files are stored in the Part 3 results folder.
"""



report_path = reports_dir / "responsible_sustainable_ai_report.md"
report_path.write_text(responsible_ai_report, encoding="utf-8")


assert (
    report_path.read_text(encoding="utf-8")
    == responsible_ai_report
)

logger.info(
    "Saved and verified reports/responsible_sustainable_ai_report.md."
)


2026-10-11 22:46:52,009 | INFO | __main__ | Saved and verified reports/responsible_sustainable_ai_report.md.
